# 09 — Lesion network mapping vs voxel-based lesion-symptom mapping
Figure 5, Supplementary figures S21 and S22. Requires the renders of 05 (Figure 1 and the robust-scale VLSM render).

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from mapstats import *
from figstyle import apply_figure_style, homogenize_fig, place_row_titles, render_info, surface_views
from plotting import corr_heatmap
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

brain, affine, _ = load_mask()
iu = np.triu_indices(6, 1)
phi = phi_matrix(impairment_labels()).values
T = {f: {k: tmap(f, k, brain) for k in KEYS} for f in (LNM, VLSM, LNM_FULL, VLSM_FULL)}
X_lv_c = matrix(lambda a, b: pearson(T[LNM][a], T[VLSM][b]), KEYS, list(KEYS)).values       # same domain on diagonal
X_lv_f = matrix(lambda a, b: pearson(T[LNM_FULL][a], T[VLSM_FULL][b]), KEYS, list(KEYS)).values
C_cc = matrix(lambda a, b: 1.0 if a == b else pearson(T[LNM][a], T[LNM][b]), KEYS).values
C_vl = matrix(lambda a, b: 1.0 if a == b else pearson(T[VLSM][a], T[VLSM][b]), KEYS).values
for M, fn in ((X_lv_c, "lnm_vs_vlsm_r"), (X_lv_f, "lnm_vs_vlsm_volnihss_r"), (C_vl, "crossdomain_pearson__vlsm")):
    pd.DataFrame(M, index=KEYS, columns=KEYS).to_csv(TAB_DIR / f"{fn}.csv")
off = ~np.eye(6, dtype=bool)
for nm, X in (("cohort only", X_lv_c), ("fully adjusted", X_lv_f)):
    print(f"LNM x VLSM, {nm}: same domain mean r = {np.diag(X).mean():.2f} ({np.diag(X).min():.2f}–{np.diag(X).max():.2f}), "
          f"different domains mean r = {X[off].mean():.2f}")
print(f"mean cross-domain r: LNM {C_cc[iu].mean():.2f}, VLSM {C_vl[iu].mean():.2f}")
for nm, (A, B_) in {"LNM ~ phi": (C_cc, phi), "VLSM ~ phi": (C_vl, phi), "LNM ~ VLSM": (C_cc, C_vl)}.items():
    rho, p = mantel(A, B_); print(f"Mantel {nm}: rho = {rho:.2f}, p = {p:.4f}")

S21: LNM x VLSM T-maps of the fully adjusted models.

In [ ]:
apply_figure_style(sizes=(9, 8, 7))
fig = corr_heatmap(X_lv_f, list(SH.values()), "LNM T-map", "VLSM T-map", "Fully adjusted models")
homogenize_fig(fig); fig.savefig(FIG_DIR / "S21_lnm_vs_vlsm_volnihss.png", dpi=300); plt.close(fig)

S22: (a) extent of the significance maps; (b) cross-domain similarity of the VLSM maps
(upper triangle Dice of pFDR < 0.05 maps, lower triangle Pearson r of unthresholded T).

In [ ]:
apply_figure_style(sizes=(8, 7, 6))
sg = {k: sigmap(VLSM, k, brain) for k in KEYS}
n_vl = np.array([sg[k].sum() for k in KEYS]); n_ln = np.array([sigmap(LNM, k, brain).sum() for k in KEYS])
MC = np.full((6, 6), np.nan)
for i, j in zip(*iu):
    MC[i, j] = dice(sg[KEYS[i]], sg[KEYS[j]]); MC[j, i] = C_vl[i, j]
print(f"VLSM cross-domain Dice {np.nanmean(MC[iu]):.2f} ± {np.nanstd(MC[iu]):.2f}, r {np.nanmean(MC.T[iu]):.2f} ± {np.nanstd(MC.T[iu]):.2f}")
C_LNM_, C_VLSM_ = "#B2182B", "#2166AC"
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(7.2, 3.1), gridspec_kw=dict(width_ratios=[1.15, 1.0], wspace=0.42))
x = np.arange(6); w = 0.38
ax_a.bar(x - w / 2, n_vl, w, color=C_VLSM_, label="VLSM")
ax_a.bar(x + w / 2, n_ln, w, color=C_LNM_, label="Label permutation-based LNM")
ax_a.set_xticks(x); ax_a.set_xticklabels([SH[k] for k in KEYS], rotation=40, ha="right")
ax_a.set_ylabel(r"Significant voxels ($p_{\mathrm{FDR}}$ < 0.05)")
ax_a.set_title("Spatial extent of significance maps", loc="left")
ax_a.legend(frameon=False, loc="upper right", fontsize=6)
ax_a.ticklabel_format(axis="y", style="plain")
cm = plt.get_cmap("viridis").copy(); cm.set_bad("0.85")
im = ax_b.imshow(MC, cmap=cm, vmin=0, vmax=1)
for i in range(6):
    for j in range(6):
        if i != j:
            v = MC[i, j]
            ax_b.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5.5, color="white" if v < 0.55 else "black")
ax_b.set_xticks(range(6)); ax_b.set_yticks(range(6))
ax_b.set_xticklabels([SH[k] for k in KEYS], rotation=40, ha="right"); ax_b.set_yticklabels([SH[k] for k in KEYS])
ax_b.tick_params(length=0)
for s_ in ax_b.spines.values(): s_.set_visible(False)
ax_b.set_title("Cross-domain similarity of VLSM maps\n(upper: Dice; lower: Pearson r)", loc="left")
cb = fig.colorbar(im, ax=ax_b, fraction=0.046, pad=0.04, extend="min", ticks=[0, 0.5, 1])
cb.ax.tick_params(labelsize=6, length=2); cb.set_label("Dice / r", fontsize=7)
fig.canvas.draw(); _inv = fig.transFigure.inverted(); _rr = fig.canvas.get_renderer()
_ytop = max(_inv.transform(ax_.get_tightbbox(_rr).get_points())[1, 1] for ax_ in (ax_a, ax_b))
for ax_, L in ((ax_a, "a"), (ax_b, "b")):
    fig.text(_inv.transform(ax_.get_tightbbox(_rr).get_points())[0, 0], _ytop + 0.01, L, fontsize=10, fontweight="bold", va="bottom")
homogenize_fig(fig); fig.savefig(FIG_DIR / "S22_vlsm_extent_and_similarity.png", dpi=300); plt.close(fig)

Figure 5. Surface views are cropped from the Figure 1 render and the robust-scale VLSM render.

In [ ]:
V_cc = surface_views(FIG_DIR / "Fig1_lnm_label_permutation.png")      # {domain: views}; n.s. domains absent
V_vl = surface_views(FIG_DIR / "vlsm_robust_scale.png")
VMAX_LNM = round(render_info(FIG_DIR / "Fig1_lnm_label_permutation.png")["vmax"])
VMAX_VLSM = round(render_info(FIG_DIR / "vlsm_robust_scale.png")["vmax"])
FULL = {k: v.lower().replace('attention / executive function', 'attention/executive') for k, v in DOMAINS.items()}
C_LNM, C_VLSM, C_SM, C_OWN = "#B2182B", "#2166AC", "#92C5DE", "#5E3C99"
SHORT = [SH[k] for k in KEYS]
DOMFULL = ["Attention / executive", "Processing speed", "Language",
           "Verbal memory", "Visuospatial functions", "Visuospatial memory"]
LETTER_X0 = 0.012                                   # shared x for left-column letters a, b, e

fig = plt.figure(figsize=(7.2, 8.2))
A_LEFT, A_TOP, A_BOTTOM = 0.035, 0.945, 0.612          # panel a: own grid, larger brains
gsA = fig.add_gridspec(1, 1, left=A_LEFT, right=0.985, top=A_TOP, bottom=A_BOTTOM)
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.92], hspace=0.50, wspace=0.55,
                      left=0.10, right=0.975, top=0.515, bottom=0.06)


# ---- (a) all six domains: two blocks of three, LNM row over VLSM row ----
ga = gsA[0, 0].subgridspec(5, 7, width_ratios=[0.30, 1, 1, 1, 1, 1, 1],
                          height_ratios=[1, 1, 0.34, 1, 1], wspace=0.02, hspace=0.04)
a_axes = []
dom_imgs = {d: [] for d in range(6)}; dom_top = {d: [] for d in range(6)}; ns_axes = []
for blk in range(2):
    for m_i, (src, lab, col) in enumerate([(V_cc, "LNM", C_LNM), (V_vl, "VLSM", C_VLSM)]):
        gr = blk * 3 + m_i
        axl = fig.add_subplot(ga[gr, 0]); axl.axis("off")
        axl.text(0.95, 0.5, lab, ha="right", va="center", fontsize=7.5, fontweight="bold",
                 color=col, transform=axl.transAxes)
        for d_i in range(3):
            d = blk * 3 + d_i
            for q_i, q in enumerate((0, 2)):                       # left lateral, right lateral
                ax = fig.add_subplot(ga[gr, 1 + 2 * d_i + q_i]); ax.axis("off"); a_axes.append(ax)
                if m_i == 0: dom_top[d].append(ax)
                if KEYS[d] not in src:                             # no significant voxels
                    ns_axes.append((d, ax)); continue
                dom_imgs[d].append(ax.imshow(src[KEYS[d]][q]))
# one horizontal colour bar per method below the maps (scales of the source renders)
_y_cb = A_BOTTOM - 0.024
for x0_, vmax, lab in [(0.27, VMAX_LNM, "LNM  T"), (0.57, VMAX_VLSM, "VLSM  T")]:
    cax = fig.add_axes([x0_, _y_cb, 0.17, 0.0075])
    cb = fig.colorbar(ScalarMappable(Normalize(-vmax, vmax), "RdBu_r"), cax=cax, orientation="horizontal",
                      extend="both", ticks=[-vmax, 0, vmax])
    cb.ax.tick_params(labelsize=5.5, length=2, pad=1)
    for t in cb.ax.get_xticklabels(): t.set_fontweight("bold")
    cb.set_label(lab, fontsize=6, fontweight="bold", labelpad=1)
ax_a_anchor = a_axes[0]

# ---- (b) own- vs other-domain map-to-map correlation ----
ax_c = fig.add_subplot(gs[0, 0])
rng = np.random.default_rng(2)
jit = {}
for x0, X in [(0, X_lv_c), (1, X_lv_f)]:
    offm = ~np.eye(6, dtype=bool)
    xo = x0 - 0.16 + rng.uniform(-0.07, 0.07, offm.sum()); xs = x0 + 0.16 + rng.uniform(-0.05, 0.05, 6)
    jit[x0] = (xo, X[offm], xs, np.diag(X))
    ax_c.scatter(xo, X[offm], s=7, color="0.7", lw=0, zorder=2)
    ax_c.scatter(xs, np.diag(X), s=16, color=C_OWN, lw=0, zorder=3)
    for xx, v in [(x0 - 0.16, np.median(X[offm])), (x0 + 0.16, np.median(np.diag(X)))]:
        ax_c.plot([xx - 0.11, xx + 0.11], [v, v], color="black", lw=1.2, zorder=4)
# annotate one same-domain dot (Language) and one different-domain dot, cohort-adjusted
li = KEYS.index("cognition-language_domain_score")
xs0, ys0 = jit[0][2][li], jit[0][3][li]
from matplotlib.patches import Ellipse
EW, EH = 0.15, 0.105                                   # ~circular at this panel's aspect
ax_c.add_patch(Ellipse((xs0, ys0), EW, EH, fill=False, ec="0.55", lw=0.7, zorder=5))
ax_c.text(xs0 + EW / 2 + 0.03, ys0 + 0.01, "Language", fontsize=5.3, color=C_OWN, ha="left", va="center")
offm = ~np.eye(6, dtype=bool); idx_pairs = np.argwhere(offm)
g = int(np.argmin(jit[0][1]))
gi, gj = idx_pairs[g]
gx, gy = jit[0][0][g], jit[0][1][g]
ax_c.add_patch(Ellipse((gx, gy), EW, EH, fill=False, ec="0.55", lw=0.7, zorder=5))
ax_c.text(-0.46, gy - EH / 2 - 0.02, f"LNM: {FULL[KEYS[gi]]}\nVLSM: {FULL[KEYS[gj]]}",
          fontsize=5.3, color="0.4", ha="left", va="top")
ax_c.set_xticks([0, 1]); ax_c.set_xticklabels(["Cohort-\nadjusted", "Fully\nadjusted"])
ax_c.set_xlim(-0.5, 1.5); ax_c.set_ylim(-0.42, 0.92); ax_c.axhline(0, color="0.85", lw=0.6, zorder=0)
ax_c.set_ylabel("Voxelwise r, LNM vs VLSM")
ax_c.scatter([], [], s=16, color=C_OWN, label="Same domain")
ax_c.scatter([], [], s=7, color="0.7", label="Different domains")
ax_c.legend(frameon=False, fontsize=5.5, loc="upper left", handletextpad=0.2, borderaxespad=0.1)

# ---- (c) correspondence matrix, cohort-adjusted ----
ax_d = fig.add_subplot(gs[0, 1])
im = ax_d.imshow(X_lv_c, vmin=-1, vmax=1, cmap="RdBu_r")
for i in range(6):
    for j in range(6):
        ax_d.text(j, i, f"{X_lv_c[i, j]:.2f}", ha="center", va="center", fontsize=4.6,
                  color="white" if abs(X_lv_c[i, j]) > 0.6 else "black")
    ax_d.add_patch(plt.Rectangle((i - .5, i - .5), 1, 1, fill=False, ec="black", lw=1.0))
ax_d.set_xticks(range(6)); ax_d.set_yticks(range(6))
ax_d.set_xticklabels(SHORT, rotation=45, ha="right", fontsize=5.3); ax_d.set_yticklabels(SHORT, fontsize=5.3)
ax_d.set_xlabel("VLSM T-map", fontsize=6, labelpad=1); ax_d.set_ylabel("LNM T-map", fontsize=6, labelpad=1)
ax_d.tick_params(length=0)
for s_ in ax_d.spines.values(): s_.set_visible(False)
cb = fig.colorbar(im, ax=ax_d, fraction=0.046, pad=0.03, extend="both", ticks=[-1, 0, 1])
cb.ax.tick_params(labelsize=5.5, length=2)
for t in cb.ax.get_yticklabels(): t.set_fontweight("bold")
cb.ax.set_title("r", fontsize=6, fontweight="bold", pad=3)

# ---- (e) similarity vs deficit co-occurrence: one facet per method, shared y ----
ge = gs[1, :].subgridspec(1, 2, wspace=0.12)
ax_e = fig.add_subplot(ge[0, 0]); ax_e2 = fig.add_subplot(ge[0, 1], sharey=ax_e)
pv = phi[iu]; yl_, yv_ = C_cc[iu], C_vl[iu]
rl, pl_ = mantel(C_cc, phi); rv, pv_ = mantel(C_vl, phi)
pair_lab = [(SH[KEYS[i]], SH[KEYS[j]]) for i, j in zip(*iu)]
def pair_idx(a, b): return next(n for n, p in enumerate(pair_lab) if set(p) == {a, b})
k_ur = pair_idx("Attn/Exec", "Language"); k_ll = pair_idx("Language", "VS mem.")
from matplotlib.patches import Ellipse
xg = np.linspace(0.13, 0.51, 100)
E_LABELS = []                       # (ax, x, y, text, all x, all y, line) -> placed after layout
for ax_, yy, col, mk, lab, rho, pp in [(ax_e, yl_, C_LNM, "o", "LNM", rl, pl_),
                                       (ax_e2, yv_, C_VLSM, "o", "VLSM", rv, pv_)]:
    b1, b0 = np.polyfit(pv, yy, 1); res = yy - (b0 + b1 * pv); n_ = len(pv)
    s_ = np.sqrt(np.sum(res ** 2) / (n_ - 2)); sxx = np.sum((pv - pv.mean()) ** 2)
    se = s_ * np.sqrt(1 / n_ + (xg - pv.mean()) ** 2 / sxx); tq = stats.t.ppf(0.975, n_ - 2)
    ax_.fill_between(xg, b0 + b1 * xg - tq * se, b0 + b1 * xg + tq * se, color=col, alpha=0.13, lw=0, zorder=0)
    ax_.plot(xg, b0 + b1 * xg, color=col, lw=1.1, zorder=2)
    ax_.scatter(pv, yy, s=20, color=col, marker=mk, zorder=3)
    ax_.text(0.03, 0.97, lab, transform=ax_.transAxes, va="top", fontsize=7, fontweight="bold", color=col)
    ax_.text(0.97, 0.04, f"\u03c1 = {rho:.2f}, Mantel p = {pp:.3f}", transform=ax_.transAxes,
             va="bottom", ha="right", fontsize=6)
    for k_, lab_ in ((k_ur, "Attn/Exec \u2013 Language"), (k_ll, "Language \u2013 VS mem.")):
        ax_.add_patch(Ellipse((pv[k_], yy[k_]), width=0.018, height=0.105, fill=False, ec="0.55", lw=0.7, zorder=4))
        E_LABELS.append((ax_, pv[k_], yy[k_], lab_, pv, yy, (b0, b1)))
    ax_.set_xlim(0.112, 0.522); ax_.set_ylim(-0.26, 1.05)
    ax_.set_xlabel("Deficit co-occurrence across patients (\u03c6)")
ax_e.set_ylabel("Cross-domain map\nsimilarity (r)")
plt.setp(ax_e2.get_yticklabels(), visible=False)

# ---- (d) LNM vs VLSM similarity structure ----
ax_f = fig.add_subplot(gs[0, 2])
ax_f.plot([-0.3, 1], [-0.3, 1], color="0.8", lw=0.7, ls="--", zorder=0)
ax_f.scatter(C_vl[iu], C_cc[iu], s=20, color="0.3", zorder=3)          # neutral: pairs, both methods
rs, ps = mantel(C_cc, C_vl)
ax_f.text(0.97, 0.03, f"\u03c1 = {rs:.2f}, Mantel p = {ps:.3f}", transform=ax_f.transAxes, va="bottom", ha="right", fontsize=6)
ax_f.set_xlim(-0.3, 1); ax_f.set_ylim(-0.3, 1); ax_f.set_aspect("equal")
# example pair (same pair as annotated in panel e)
k_ex = pair_idx("Attn/Exec", "Language")
ex_x, ex_y = C_vl[iu][k_ex], C_cc[iu][k_ex]
ax_f.add_patch(Ellipse((ex_x, ex_y), 0.11, 0.11, fill=False, ec="0.55", lw=0.7, zorder=5))
ax_f.text(ex_x - 0.075, ex_y, "Attn/Exec \u2013 Language", ha="right", va="center", fontsize=5.5, color="0.3")
ax_f.set_xlabel("VLSM similarity (r)"); ax_f.set_ylabel("LNM similarity (r)")

# ---- aligned subtitles + letters ----
LX = {"a": LETTER_X0, "b": LETTER_X0, "e": LETTER_X0}
fig.canvas.draw()
ya = max(ax.get_position().y1 for ax in a_axes) + 0.026
fig.text(LETTER_X0, ya, "a", fontsize=10, fontweight="bold", va="baseline")
fig.text(LETTER_X0 + 0.028, ya, "Cohort-adjusted LNM and VLSM T-maps", fontsize=7.5, va="baseline")
fig.canvas.draw()
_tb = fig.transFigure.inverted().transform(ax_d.get_tightbbox(fig.canvas.get_renderer()).get_points())
LX["c"] = _tb[0, 0] + 0.035                               # nudge right of the rotated tick labels
place_row_titles(fig, [(ax_c, "b", "Map-to-map correspondence"),
                       (ax_d, "c", "LNM \u00d7 VLSM T-maps"),
                       (ax_f, "d", "Cross-domain similarity")], letter_x=LX, title_gap=0.028)
place_row_titles(fig, [(ax_e, "e", "Cross-domain similarity vs deficit co-occurrence")], letter_x=LX, title_gap=0.028)

# domain headers + n.s. centred between the two brains of each domain (figure coords)
inv = fig.transFigure.inverted(); rr = fig.canvas.get_renderer()
dom_xc = {}
for d in range(6):
    ext = [inv.transform(im.get_window_extent(rr).get_points()) for im in dom_imgs[d]]
    dom_xc[d] = (min(e[0, 0] for e in ext) + max(e[1, 0] for e in ext)) / 2
    ytop = max(a.get_position().y1 for a in dom_top[d])
    fig.text(dom_xc[d], ytop + 0.004, DOMFULL[d], ha="center", va="bottom", fontsize=6.5)
for d, a_ in ns_axes[:1]:
    p_ = a_.get_position()
    fig.text(dom_xc[d], (p_.y0 + p_.y1) / 2, "n.s.", ha="center", va="center", fontsize=7,
             fontstyle="italic", color="0.45")

# ---- panel e: collision-aware placement of the example labels (after the layout is final) ----
import matplotlib.transforms as mtrans
fig.canvas.draw()
if fig.get_layout_engine() is not None: fig.set_layout_engine("none")
_r = fig.canvas.get_renderer()
_PT = fig.dpi / 72.0
def _place_label(ax_, x, y, txt, X, Y, line, placed):
    pts = ax_.transData.transform(np.c_[X, Y])
    xl = np.linspace(*ax_.get_xlim(), 300); ln = ax_.transData.transform(np.c_[xl, line[0] + line[1] * xl])
    axbb = ax_.get_window_extent(_r)
    others = [t.get_window_extent(_r) for t in ax_.texts if t.get_text().strip()] + placed
    rings = [p.get_window_extent(_r) for p in ax_.patches]
    best = None
    for variant in (txt, txt.replace(" \u2013 ", " \u2013\n")):
        for ang in range(0, 360, 15):
            for dist in (6, 9, 14):
                c, s_ = np.cos(np.radians(ang)), np.sin(np.radians(ang))
                ha = "left" if c > 0.3 else ("right" if c < -0.3 else "center")
                va = "bottom" if s_ > 0.3 else ("top" if s_ < -0.3 else "center")
                tr = mtrans.offset_copy(ax_.transData, fig=fig, x=dist * c, y=dist * s_, units="points")
                t = ax_.text(x, y, variant, transform=tr, ha=ha, va=va, fontsize=5.5, color="0.3",
                             fontweight="bold", linespacing=1.0, zorder=6)
                bb = t.get_window_extent(_r)
                pad = 3.2 * _PT                                     # dot radius + margin
                bbp = mtrans.Bbox.from_extents(bb.x0 - pad, bb.y0 - pad, bb.x1 + pad, bb.y1 + pad)
                n_dot = sum(bbp.contains(*p) for p in pts)
                bbl = mtrans.Bbox.from_extents(bb.x0 - 1.5 * _PT, bb.y0 - 1.5 * _PT, bb.x1 + 1.5 * _PT, bb.y1 + 1.5 * _PT)
                n_line = sum(bbl.contains(*p) for p in ln)
                n_ring = sum(bb.overlaps(rb) for rb in rings)
                n_txt = sum(bb.overlaps(ob) for ob in others)
                outside = not (axbb.x0 + 2 <= bb.x0 and bb.x1 <= axbb.x1 - 2 and axbb.y0 + 2 <= bb.y0 and bb.y1 <= axbb.y1 - 2)
                tgt = ax_.transData.transform((x, y))
                _d = lambda p: np.hypot(max(bb.x0 - p[0], 0, p[0] - bb.x1), max(bb.y0 - p[1], 0, p[1] - bb.y1))
                d_t = _d(tgt); d_o = min(_d(p) for p in pts if np.hypot(*(p - tgt)) > 1)
                ambiguous = d_o < d_t + 2 * _PT                      # another dot (almost) as close as the target
                score = 20 * n_dot + (6 + 0.05 * n_line if n_line else 0) + 15 * n_ring + 40 * n_txt + 60 * outside + 12 * ambiguous \
                        + 0.1 * dist + (1.0 if "\n" in variant else 0)
                if best is None or score < best[0]:
                    if best is not None: best[1].remove()
                    best = (score, t, bb)
                else:
                    t.remove()
    placed.append(best[2])
    return best[0]
_placed = {}
E_SCORES = [_place_label(*e, _placed.setdefault(id(e[0]), [])) for e in E_LABELS]
homogenize_fig(fig)
fig.savefig(FIG_DIR / "Fig5_lnm_vs_vlsm.png", dpi=300, bbox_inches=None)
fig.savefig(FIG_DIR / "Fig5_lnm_vs_vlsm.pdf", bbox_inches=None)